# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided example for loading and exploring the FAIR² protein dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/) accessible via a public URL.

In [ ]:
# Install `mlcroissant` package if not already available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(metadata.name)
print(metadata.description)

## 2. Data Overview
Review available record sets and their field IDs.

We list the available `RecordSet` entities in the metadata, along with their `@id` and field IDs. For this dataset, let's inspect the high-level structure first, and retrieve example records from each set.

In [ ]:
# List all available record sets, their @id, and the fields they provide
for record_set in metadata.record_sets:
    print(f"RecordSet Name: {record_set.name}")
    print(f"@id: {record_set.id}")
    field_ids = [field.id for field in record_set.fields]
    print(f"Field @ids: {field_ids}")
    print('-'*60)

# Display some example records for each RecordSet
for record_set in metadata.record_sets:
    print(f"Example records from RecordSet: {record_set.name} (id: {record_set.id})")
    records = list(dataset.records(record_set=record_set.id))
    # Show up to 2 records, or less if not available
    for record in records[:2]:
        print(record)
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll use the main protein record set (named typically as 'protein abundance', 'table', or similar). Identify the primary record sets from above, and collect their `@id`.

We'll extract all such tables, but focus on the major quantitative one for further work.

In [ ]:
# Collect all RecordSet @ids
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}

# Load records as DataFrames for each record set
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet {record_set_id}, columns:")
        print(df.columns.tolist())
        print()
    else:
        print(f"No records found in RecordSet {record_set_id}\n")

# We'll use the first non-empty RecordSet for analysis below:
main_record_set_id = None
for record_set_id in record_sets:
    if record_set_id in dataframes and len(dataframes[record_set_id]) > 0:
        main_record_set_id = record_set_id
        break

if main_record_set_id is not None:
    print(f"Columns in selected main RecordSet ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform some simple analysis: filtering records, normalizing numeric fields, examining grouped means, etc.

- Select a representative numeric field (e.g., 'abundance', 'MW', or similar) for filtering and normalization.
- Optionally, group by a categorical field (e.g., 'sample', 'protein_class', etc.) if present.

In [ ]:
# Choose a numeric field for demonstration (adjust this per actual data columns)
df = dataframes[main_record_set_id]
# Guess a candidate numeric field from typical names
possible_numeric_fields = [col for col in df.columns if any(k in col.lower() for k in ["abundance", "mw", "peptide", "coverage", "score"])]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    # If not found, pick first float/integer column
    numeric_field = df.select_dtypes(include=['number']).columns[0]

print(f"Using numeric field: '{numeric_field}'")
# Filter, e.g., greater than mean
threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows out of {len(df)}")
display(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field, e.g., 'sample' or first string category
possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    print(f"Grouped means by category in: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships.

Let's plot the distribution of the filtered numeric field, and optionally, its average by a group category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field], kde=True)
plt.title(f"Distribution of {numeric_field} (filtered)")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot for group, if available
if group_field:
    plt.figure(figsize=(12,4))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.xticks(rotation=45, ha='right')
    plt.title(f"{numeric_field} by {group_field} (filtered records)")
    plt.show()

## 6. Conclusion
- We loaded the FAIR² dataset and explored all available data tables (record sets) using their Croissant schema `@id`s.
- We identified and analyzed protein abundance or molecular weight data, filtered, normalized, and grouped values for quick insight.
- The notebook demonstrates how to navigate Croissant-formatted datasets by machine-readable identifiers, which supports robust, programmatic data science workflows.

Continue by adapting filters and visualizations for your specific scientific questions!